# Stability — feature-importance distance, parameter distance

Bootstraps each model to measure how much its feature importances, parameters, and
performance shift under resampled training data, for whichever models have a
`models/<name>_model.py` file so far.

**Compute note**: bootstrapping refits a model `N_BOOT` times. If a single fit is
slow (especially for a heavily-tuned XGBoost or a fine-tuning TabPFN), lower
`N_BOOT` or subsample `X_train` before bootstrapping — this is a scope call, not a
correctness one.

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path("../..").resolve()))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score

from common_metrics import (
    FEATURES, MODEL_NAMES, TEAM_THRESHOLD,
    load_split, get_X_y, available_models, report_status, importance_vector, get_or_fit_model,
)

report_status()

## Load data

In [ ]:
train_df = load_split("train")
test_df = load_split("test")

X_train, y_train = get_X_y(train_df)
X_test, y_test = get_X_y(test_df)

print(f"train: {X_train.shape}, test: {X_test.shape}")

## Feature-importance distance + performance stability — one bootstrap, both numbers

`d(f1, f2) = ||φ(f1) - φ(f2)||₂` — model-agnostic via `common_metrics.importance_vector`,
so this is the one number comparable across xgboost/logreg/tabpfn. Bundled with the AUC
spread in the same loop: both need the same bootstrap resamples, so one fit per iteration
produces both numbers instead of fitting twice for the same information.

**TabPFN nuance**: if TabPFN's own `fit()` re-subsamples its ~10K-row context on
every call, bootstrapping here (which resamples `X_train` before calling `fit()`)
tests sensitivity to *which* subsample gets drawn — that's the more informative
question for this dataset's scale. If instead `fit()` uses a fixed, pre-chosen
context, this only tests ordinary sampling noise. Confirm which applies once the
TabPFN owner's module lands.

In [ ]:
def stability_bootstrap(name, module, model_type, X_train, y_train, X_test, y_test, n_boot=30, random_state=42):
    rng = np.random.RandomState(random_state)
    base_model = get_or_fit_model(name, module, X_train, y_train)  # the real model if saved, else a fresh fit
    base_imp = importance_vector(model_type, base_model, module, X_train, y_train)

    imp_distances, aucs = [], []
    for i in range(n_boot):
        idx = rng.choice(len(X_train), size=len(X_train), replace=True)
        X_b, y_b = X_train.iloc[idx], y_train.iloc[idx]
        model_b = module.fit(X_b, y_b)  # resampled data has no saved artifact — must fit
        imp_b = importance_vector(model_type, model_b, module, X_b, y_b)
        imp_distances.append(float(np.linalg.norm(base_imp - imp_b)))
        probs = module.predict_proba(model_b, X_test)[:, 1]
        aucs.append(roc_auc_score(y_test, probs))
    return imp_distances, aucs

## Parameter distance — logreg only

In [ ]:
def parameter_distance_bootstrap(name, module, X_train, y_train, n_boot=30, random_state=42):
    rng = np.random.RandomState(random_state)
    base_model = get_or_fit_model(name, module, X_train, y_train)
    theta_base = np.asarray(base_model.coef_[0])
    distances = []
    for i in range(n_boot):
        idx = rng.choice(len(X_train), size=len(X_train), replace=True)
        X_b, y_b = X_train.iloc[idx], y_train.iloc[idx]
        theta_b = np.asarray(module.fit(X_b, y_b).coef_[0])
        distances.append(float(np.linalg.norm(theta_base - theta_b)))
    return distances

## D1 → D2 — does the model change as it sees more data?

The syllabus's own worked example (§7.1): two datasets of different size (D1 smaller,
D2 = the full set), compare feature importances directly — no resampling. This answers a
different question than the bootstrap above: bootstrap asks "how much does the model wobble
from sampling noise at a *fixed* size," this asks "has the model actually converged, or would
more data still change it." Only 2 fits, far cheaper than the bootstrap.

**TabPFN caveat**: if its `fit()` internally subsamples to its own fixed context size
regardless of how much data it's handed, D1 and D2 may collapse to the same effective
context, making this comparison uninformative for that model specifically.

In [ ]:
def stability_d1_d2(name, module, model_type, X_train, y_train, frac_d1=0.5, random_state=42):
    rng = np.random.RandomState(random_state)
    idx_d1 = rng.choice(len(X_train), size=int(len(X_train) * frac_d1), replace=False)
    X_d1, y_d1 = X_train.iloc[idx_d1], y_train.iloc[idx_d1]

    model_d1 = module.fit(X_d1, y_d1)  # D1 (50%) has no saved artifact — must fit
    model_d2 = get_or_fit_model(name, module, X_train, y_train)  # D2 (100%) is exactly what the saved model was trained on

    imp_d1 = importance_vector(model_type, model_d1, module, X_d1, y_d1)
    imp_d2 = importance_vector(model_type, model_d2, module, X_train, y_train)
    return float(np.linalg.norm(imp_d1 - imp_d2))

## Run across available models

In [ ]:
MODEL_TYPE = {"xgboost": "xgboost", "logreg": "logreg", "tabpfn": "tabpfn"}
N_BOOT = 30  # lower this (or subsample X_train) if a single fit is slow

stability_results = {}
for name, module in available_models().items():
    print(f"\n=== {name} ===")
    try:
        model_type = MODEL_TYPE[name]
        imp_distances, auc_spread = stability_bootstrap(name, module, model_type, X_train, y_train, X_test, y_test, n_boot=N_BOOT)
        d1_d2_distance = stability_d1_d2(name, module, model_type, X_train, y_train)
        stability_results[name] = {
            "importance_distance": imp_distances,
            "auc_spread": auc_spread,
            "d1_d2_distance": d1_d2_distance,
        }
        print(f"  importance distance (bootstrap): mean={np.mean(imp_distances):.4f}, std={np.std(imp_distances):.4f}")
        print(f"  AUC spread: mean={np.mean(auc_spread):.4f}, std={np.std(auc_spread):.4f}")
        print(f"  D1→D2 importance distance (50% → 100% of data): {d1_d2_distance:.4f}")

        if name == "logreg":
            param_distances = parameter_distance_bootstrap(name, module, X_train, y_train, n_boot=N_BOOT)
            stability_results[name]["parameter_distance"] = param_distances
            print(f"  parameter distance: mean={np.mean(param_distances):.4f}, std={np.std(param_distances):.4f}")
    except Exception as e:
        print(f"  {name} failed ({type(e).__name__}: {e}) — skipping")
        continue

if not stability_results:
    print("No models ready yet — drop a models/<name>_model.py file in and re-run.")

## Plot — importance-distance distribution per model

In [ ]:
if stability_results:
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.boxplot(
        [r["importance_distance"] for r in stability_results.values()],
        tick_labels=list(stability_results.keys()),
    )
    ax.set_ylabel("feature-importance distance (bootstrap)")
    plt.tight_layout()
    plt.show()

## Smoke test — remove once real models are in `models/`

Same throwaway logistic regression as the interpretability notebook, on a small
sample and few bootstrap iterations, purely to check the harness runs.

In [ ]:
from sklearn.linear_model import LogisticRegression

class _SmokeTestModule:
    _medians = None  # fixed at fit time so a later all-NaN batch (e.g. a masked coalition) still fills

    @staticmethod
    def fit(X, y):
        Xn = X.select_dtypes("number")
        _SmokeTestModule._medians = Xn.median().fillna(0)
        return LogisticRegression(max_iter=200).fit(Xn.fillna(_SmokeTestModule._medians), y)

    @staticmethod
    def predict_proba(model, X):
        Xn = X.select_dtypes("number").fillna(_SmokeTestModule._medians)
        return model.predict_proba(Xn)

if not stability_results:
    print("Running a throwaway smoke test — NOT a real model, just checking the harness works end to end.")
    sample = train_df.sample(20_000, random_state=42)
    Xs, ys = get_X_y(sample)
    test_sample = test_df.sample(2_000, random_state=42)
    Xt, yt = get_X_y(test_sample)
    smoke_distances = stability_bootstrap(_SmokeTestModule, "logreg", Xs, ys, n_boot=5)
    print("Smoke test importance distances:", smoke_distances, "— harness is wired correctly.")